# Phase 2: Image Processing Pipeline Demonstration
### Project: Automated Classification of Martian Surface Images Captured by NASA's Curiosity Rover Using Machine Learning

This notebook demonstrates the standardized, deterministic image preprocessing pipeline defined in `preprocessing/image_processing.py`:
1. **Image Loading**: Reading browse JPEGs.
2. **Channel Standardization**: Replicating 1-channel Grayscale images to 3-channel RGB.
3. **Resizing Strategy**: Evaluating Direct Resizing vs. Aspect-Ratio-Preserving Letterboxing.
4. **Pixel Normalization**: Scaling uint8 [0, 255] to float32 [0.0, 1.0].

> **Production Decision**: Aspect-Ratio-Preserving Letterboxing with `pad_mode="constant"` (neutral black border, `pad_val=0`) is selected as the production standard to prevent geometric distortion of Martian geology.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on sys.path
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import config
from preprocessing.image_processing import (
    load_image,
    standardize_channels,
    resize_image,
    normalize_pixels,
    preprocess_image
)

## 1. Comparing Resizing Strategies: Direct Stretching vs. Letterboxing
Many Martian surface images have native dimensions of ~256x192 (aspect ratio ~1.33). Directly stretching to 256x256 distorts rock grain and circular rover drill holes.

In [ ]:
# Select a non-square sample image
sample_file = config.CALIBRATED_IMG_DIR / "0077ML0005780000102730I01_DRCL.JPG"

raw_img = load_image(sample_file)
rgb_img = standardize_channels(raw_img)
direct_resized = resize_image(rgb_img, target_size=(256, 256), preserve_aspect_ratio=False)
letterbox_resized = resize_image(rgb_img, target_size=(256, 256), preserve_aspect_ratio=True, pad_mode="constant", pad_value=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(rgb_img)
axes[0].set_title(f"Original ({rgb_img.shape[1]}x{rgb_img.shape[0]})")

axes[1].imshow(direct_resized)
axes[1].set_title("Direct Resize (Distorted: 256x256)")

axes[2].imshow(letterbox_resized)
axes[2].set_title("Letterboxed (Production: 256x256)")
plt.tight_layout()
plt.show()

## 2. End-to-End Preprocessing on Grayscale vs. RGB Images
Verifying that the unified `preprocess_image` pipeline seamlessly processes both RGB and Grayscale images to identical `(256, 256, 3)` float32 tensors in `[0.0, 1.0]`.

In [ ]:
rgb_file = config.CALIBRATED_IMG_DIR / "0077ML0005780000102730I01_DRCL.JPG"
gray_file = config.CALIBRATED_IMG_DIR / "0013MR0000020060100034D01_DRCL.JPG"

processed_rgb = preprocess_image(rgb_file, target_size=(256, 256), preserve_aspect_ratio=True)
processed_gray = preprocess_image(gray_file, target_size=(256, 256), preserve_aspect_ratio=True)

print(f"Processed RGB shape : {processed_rgb.shape}, dtype: {processed_rgb.dtype}, range: [{processed_rgb.min():.3f}, {processed_rgb.max():.3f}]")
print(f"Processed Gray shape: {processed_gray.shape}, dtype: {processed_gray.dtype}, range: [{processed_gray.min():.3f}, {processed_gray.max():.3f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(processed_rgb)
axes[0].set_title("Preprocessed RGB Image")
axes[1].imshow(processed_gray)
axes[1].set_title("Preprocessed Grayscale Image (Harmonized to RGB)")
plt.tight_layout()
plt.show()